# Example end-to-end scenario: preprocess → BESS dispatch → KPIs & plots

This notebook runs a minimal reproducible example for one scenario using synthetic sample data so reviewers can reproduce the workflow. It demonstrates: preprocessing (harmonising to hourly), the Chapter‑4 rule‑based dispatch, KPI calculation, and simple plots.

Notes: this is a small example using synthetic data (24 hourly steps). Replace the sample inputs with real Silver Parquet artifacts (silver/load_hourly.parquet, silver/pv_hourly.parquet, silver/prices_hourly.parquet) for full runs.

In [ ]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass

%matplotlib inline


In [ ]:
# 1) Create synthetic sample data (24 hours)
idx = pd.date_range('2025-06-21', periods=24, freq='H', tz='Europe/Madrid')
np.random.seed(0)
# simple daily load profile with evening peak
base_load = 1.5 + 0.5 * np.sin((idx.hour - 7) / 24 * 2 * np.pi)
evening = 1.0 * ((idx.hour >= 18) & (idx.hour <= 22)).astype(float)
load_kwh = (base_load + evening) * 1.5  # scale to kWh values
# PV: no generation at night, peak midday
pv_kwh = np.maximum(0, 3.0 * np.sin((idx.hour - 6) / 24 * 2 * np.pi))
# Prices: simple two-level price (high in evening)
price_eur_kwh = np.where((idx.hour >= 18) & (idx.hour <= 22), 0.35, 0.20)
export_credit_eur_kwh = 0.05  # simplified export credit

df = pd.DataFrame({'load_kwh': load_kwh, 'pv_kwh': pv_kwh, 'price_eur_kwh': price_eur_kwh}, index=idx)
df.head()


In [ ]:
# 2) Battery parameters and dispatch function (rule-based, Chapter 4 style)
@dataclass
class BESSParams:
    E_nom: float  # kWh
    S_min: float  # kWh
    S_max: float  # kWh
    P_ch_max: float  # kW (≈ kWh per hour)
    P_dis_max: float
    eta_ch: float
    eta_dis: float
    omie_threshold: float = None
    use_omie: bool = False

def dispatch_series(df, params):
    # prepare result containers
    S = params.S_max * 0.5  # start at 50% SoC
    rows = []
    for ts, row in df.iterrows():
        Load_t = float(row['load_kwh'])
        PV_t = float(row['pv_kwh'])
        # 1) PV to load
        pv_to_load = min(PV_t, Load_t)
        load_remaining = Load_t - pv_to_load
        pv_remaining = PV_t - pv_to_load
        # 2) charge from PV only
        P_ch = 0.0
        if pv_remaining > 0 and S < params.S_max:
            max_charge_energy = min(pv_remaining, params.P_ch_max, (params.S_max - S) / params.eta_ch)
            P_ch = max_charge_energy
        # 3) discharge to meet load remaining
        P_dis = 0.0
        if load_remaining > 0 and S > params.S_min:
            max_discharge_energy = min(load_remaining, params.P_dis_max, (S - params.S_min) * params.eta_dis)
            P_dis = max_discharge_energy
            load_remaining -= P_dis
        # OMIE optional omitted in this simple example
        # 5) update SoC
        S_new = S + params.eta_ch * P_ch - P_dis / params.eta_dis
        S_new = max(min(S_new, params.S_max), params.S_min)
        # 6) net flow and imports/exports
        net = Load_t - PV_t - P_dis + P_ch
        Import = max(net, 0.0)
        Export = max(-net, 0.0)
        rows.append({'timestamp': ts, 'P_ch': P_ch, 'P_dis': P_dis, 'SoC': S_new, 'Import_kwh': Import, 'Export_kwh': Export})
        S = S_new
    out = pd.DataFrame(rows).set_index('timestamp')
    return out

# define params and run
params = BESSParams(E_nom=50.0, S_min=5.0, S_max=50.0, P_ch_max=25.0, P_dis_max=25.0, eta_ch=0.95, eta_dis=0.95)
res = dispatch_series(df, params)
res.head()


In [ ]:
# 3) Combine results and compute KPIs
out = df.join(res)
# basic KPIs for the sample horizon
total_load = out['load_kwh'].sum()
total_pv = out['pv_kwh'].sum()
total_import = out['Import_kwh'].sum()
total_export = out['Export_kwh'].sum()
self_consumed_pv = total_pv - total_export
self_consumption_ratio = self_consumed_pv / total_pv if total_pv>0 else np.nan
self_sufficiency = (total_load - total_import) / total_load
energy_shifted = out['P_dis'].sum()  # kWh discharged to meet load
# simple bill calculation: import cost - export credit
import_cost = (out['Import_kwh'] * out['price_eur_kwh']).sum()
export_credit = (out['Export_kwh'] * export_credit_eur_kwh).sum()
net_bill = import_cost - export_credit
kpi = {
    'total_load_kwh': total_load,
    'total_pv_kwh': total_pv,
    'total_import_kwh': total_import,
    'total_export_kwh': total_export,
    'SCR': self_consumption_ratio,
    'SSR': self_sufficiency,
    'energy_shifted_kwh': energy_shifted,
    'import_cost_eur': import_cost,
    'export_credit_eur': export_credit,
    'net_bill_eur': net_bill
}
pd.Series(kpi)


In [ ]:
# 4) Simple plots
plt.figure(figsize=(10,5))
plt.plot(out.index, out['load_kwh'], label='Load (kWh)', marker='o')
plt.plot(out.index, out['pv_kwh'], label='PV (kWh)', marker='o')
plt.plot(out.index, out['SoC'], label='SoC (kWh)', marker='o')
plt.legend()
plt.xticks(rotation=45)
plt.title('Load, PV and Battery SoC')
plt.show()

plt.figure(figsize=(8,3))
plt.bar(out.index, out['Import_kwh'], label='Import kWh')
plt.bar(out.index, out['Export_kwh'], bottom=0, label='Export kWh')
plt.xticks(rotation=45)
plt.title('Imports and Exports')
plt.legend()
plt.show()


---
Next steps for reviewers: replace the synthetic data with real Silver artifacts (silver/load_hourly.parquet, silver/pv_hourly.parquet, silver/prices_hourly.parquet) and rerun. Adjust BESSParams to match scenario definitions.
